In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

print('Libraries loaded successfully')

In [ ]:
df = pd.read_csv('airline_satisfaction.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
df = df.dropna()
print('Shape after cleaning:', df.shape)
print('\nSatisfaction Value Counts:')
print(df['satisfaction'].value_counts())

In [ ]:
df['satisfaction_encoded'] = df['satisfaction'].map({'satisfied': 1, 'dissatisfied': 0})
df['Customer Type_encoded'] = df['Customer Type'].map({'Loyal Customer': 1, 'disloyal Customer': 0})
df['Type of Travel_encoded'] = df['Type of Travel'].map({'Business travel': 1, 'Personal Travel': 0})
df['Class_encoded'] = df['Class'].map({'Business': 2, 'Eco Plus': 1, 'Eco': 0})
print('Encoding complete!')
print(df[['satisfaction', 'satisfaction_encoded']].head())

In [ ]:
plt.figure(figsize=(8, 5))
df['satisfaction'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'], edgecolor='black')
plt.title('Passenger Satisfaction Distribution')
plt.xlabel('Satisfaction')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
numeric_features = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
df[numeric_features].hist(figsize=(12, 6), bins=30, color='steelblue', edgecolor='black')
plt.suptitle('Distribution of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Class', hue='satisfaction', palette=['salmon', 'steelblue'])
plt.title('Satisfaction by Travel Class')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Customer Type', hue='satisfaction', palette=['salmon', 'steelblue'])
plt.title('Satisfaction by Customer Type')
plt.xlabel('Customer Type')
plt.ylabel('Count')
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)
plt.figure(figsize=(14, 10))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [
    'Customer Type_encoded', 'Age', 'Type of Travel_encoded', 'Class_encoded',
    'Flight Distance', 'Seat comfort', 'Inflight wifi service',
    'Inflight entertainment', 'Online support', 'Ease of Online booking',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Cleanliness', 'Online boarding',
    'Departure Delay in Minutes', 'Arrival Delay in Minutes'
]
X = df[feature_cols]
y = df['satisfaction_encoded']
vif_data = pd.DataFrame()
vif_data['Feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print('=== VIF Multicollinearity Check ===')
print(vif_data.sort_values('VIF', ascending=False))

In [ ]:
X_const = sm.add_constant(X)
logit_model = sm.Logit(y, X_const).fit()
print(logit_model.summary())

In [ ]:
print('=== MODEL SIGNIFICANCE ===')
print(f'Log-Likelihood: {logit_model.llf:.4f}')
print(f'AIC: {logit_model.aic:.4f}')
print(f'BIC: {logit_model.bic:.4f}')
print(f'Pseudo R-squared: {logit_model.prsquared:.4f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Dissatisfied', 'Satisfied'],
            yticklabels=['Dissatisfied', 'Satisfied'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
print('=== LINEARITY CHECK ===')
for col in feature_cols:
    corr, pval = stats.pearsonr(df[col], y)
    print(f'  {col}: r={corr:.4f}, p-value={pval:.4f}')

In [ ]:
residuals = logit_model.resid_response
fitted = logit_model.fittedvalues
plt.figure(figsize=(8, 5))
plt.scatter(fitted, residuals, alpha=0.3, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Q-Q Plot of Residuals')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals')
plt.show()

In [ ]:
print('=== LOGISTIC REGRESSION INTERPRETATION ===')
print(f'Pseudo R-squared: {logit_model.prsquared:.4f}')
print(f'This means the model explains {logit_model.prsquared*100:.2f}% of variance in satisfaction.')
print()
print('=== P-VALUES (STATISTICAL SIGNIFICANCE) ===')
for name, pval in zip(X_const.columns, logit_model.pvalues):
    sig = 'Significant' if pval < 0.05 else 'Not Significant'
    print(f'  {name}: p-value={pval:.4f} --> {sig}')
print()
print('=== 95% CONFIDENCE INTERVALS ===')
print(logit_model.conf_int())
print()
print('=== COEFFICIENT INTERPRETATION ===')
for name, coef in zip(X.columns, logit_model.params[1:]):
    print(f'  {name}: holding all other variables constant, a one-unit increase changes log-odds of satisfaction by {coef:.4f}')
print()
print('=== MODEL ACCURACY ===')
print(f'Overall Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print()
print('=== BUSINESS INSIGHTS ===')
print('1. Inflight wifi service and online boarding are strong predictors of satisfaction.')
print('2. Business class passengers tend to be more satisfied than Eco passengers.')
print('3. Loyal customers show higher satisfaction rates.')
print('4. Departure and arrival delays negatively impact satisfaction.')
print()
print('=== RECOMMENDATIONS ===')
print('- Invest in improving inflight wifi and entertainment services.')
print('- Reduce departure and arrival delays to boost satisfaction.')
print('- Focus on loyalty programs to retain satisfied customers.')
print('- Improve online boarding experience for all travel classes.')